In [1]:
# imports
import json
import os
import numpy as np
from tqdm import tqdm
from google import genai
from google.genai import types
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai_client = genai.Client(api_key=GOOGLE_API_KEY)

COLLECTION_NAME = "docs_api"
VECTOR_SIZE = 768

qdrant = QdrantClient(url="http://localhost:6333")

##### Create Collection (if not exists)

In [4]:
collections = qdrant.get_collections().collections
existing = [c.name for c in collections]

if COLLECTION_NAME not in existing:
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE
        )
    )
    print("Collection created:", COLLECTION_NAME)
else:
    print("Collection already exists:", COLLECTION_NAME)


Collection already exists: docs_api


##### Embed and Upload to Qdrant

In [2]:
def embed_texts(texts):
    res = genai_client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts,
        config=types.EmbedContentConfig(output_dimensionality=VECTOR_SIZE),
    )
    return [e.values for e in res.embeddings]

def normalize(vec):
    vec = np.array(vec, dtype=float)
    norm = np.linalg.norm(vec)
    if norm == 0:
        return vec
    return (vec / norm).tolist()

In [ ]:
with open("text_to_embed.json", "r", encoding="utf-8") as f:
    data = json.load(f)

texts = [d["text"] for d in data]

# Embed in batches
BATCH = 64
points = []

print("Embedding & uploading to Qdrant...")

for i in range(0, len(texts), BATCH):
    batch = data[i:i+BATCH]
    batch_text = [b["text"] for b in batch]

    vectors = embed_texts(batch_text)

    for idx, (item, vector) in enumerate(zip(batch, vectors)):
        vector = normalize(vector)
        p = PointStruct(
            id=i + idx,
            vector=vector,
            payload={
                "text": item["text"],
                "page_title": item.get("page_title", ""),
                "section_title": item.get("section_title", ""),
                "url": item.get("url", ""),
            }
        )
        points.append(p)

    # Upload batch
    qdrant.upsert(
        collection_name=COLLECTION_NAME,
        points=points
    )
    points = []

print("Done uploading.")

Embedding & uploading to Qdrant...
Done uploading.


In [4]:
info = qdrant.get_collection(collection_name="docs_api")
count = info.points_count

print("Total points in collection:", count)

Total points in collection: 86


##### Query + Return Top Chunks

In [5]:
def embed_query(query: str):
    result = genai_client.models.embed_content(
        model="gemini-embedding-001",
        contents=[query],
        config=types.EmbedContentConfig(output_dimensionality=VECTOR_SIZE)
    )
    return result.embeddings[0].values


In [6]:
def search(query, top_k=10):
    qvec = normalize(embed_query(query))

    res = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=qvec,
        limit=top_k,
    )

    points = getattr(res, "points", res)

    results = []
    for r in points:
        results.append({
            "id": r.id,
            "score": r.score,
            "text": r.payload.get("text", ""),
            "page_title": r.payload.get("page_title", ""),
            "section_title": r.payload.get("section_title", ""),
            "url": r.payload.get("url", "")
        })

    return results


In [7]:
# Test
query = "users"
matches = search(query, top_k=10)

for i, m in enumerate(matches, 1):
    print(f"\n### Result {i}")
    print("Score:", m["score"])
    print(m["text"])


### Result 1
Score: 0.5637659
Security File: This section ensures user rights are safeguarded. It covers privacy policies, data protection measures, and guidelines for responsible usage, promoting a secure environment for all.

### Result 2
Score: 0.562275
Ensuring Safety: The application ensures security through encryption, secure login credentials, and regular updates. We prioritize user privacy and conduct security audits to maintain a safe learning environment for all users.

### Result 3
Score: 0.5417682
Platform Audience: Built for Every Role in Your School Community. Our platform adapts to meet the unique needs of students, parents, teachers, and administrators, creating a unified educational ecosystem. Students, Smart Learners. Access your lessons, exams, and school updates from anywhere. Learn, collaborate, and stay informed through a personalized platform. Features: Course Materials & Videos, Timetable & Agenda View, Online Exams & Results, Messaging with Teachers Parents, I